# EgoEvGesture — VideoMamba-Tiny · TRAIN + VALIDATION · Resume Across Kaggle Sessions

**Objective:** Train a **38-class** classifier using the published Heterogeneous Cleaned Split v1. **Do not use `test.csv`** for training or checkpoint selection. Event `.npy` arrays have shape `(N, 4)` with columns **`x, y, polarity, timestamp`** (zero-based timestamp column 3, as in the official EgoEvGesture code).

This standalone notebook writes `checkpoint-latest.pth`, `checkpoint-best.pth`, `training_history.csv`, and `experiment_config.json` to `/kaggle/working/egoev_videomamba_tiny_hetero_v1/`. Each session trains for up to `EPOCHS_THIS_SESSION` additional epochs, resuming until `TOTAL_EPOCHS` is reached.

**Before running, add these Kaggle Inputs:**

1. `hoanganh94/egoevgesture-raw` — original event data.
2. `hoanganh94/egoevgesture-heterogeneous-split-dataset` — includes `egoev_heterogeneous_split_v1/`.
3. `hoanganh94/videomamba-t4-cuda-wheels` — **if** `mamba_ssm` and `causal_conv1d` are not installed; wheel versions must match the current Python, PyTorch, and CUDA environment.
4. **From session two onward:** Add Input from the previous training session's saved output, containing **both** `checkpoint-latest.pth` and `checkpoint-best.pth`.
5. **First session:** Enable Internet to retrieve VideoMamba source and pretrained weights, or add these artifacts through Kaggle Input.

**Important:** Kaggle `/kaggle/working` is not guaranteed to persist across separate runtimes. Save the completed session's outputs as a notebook version or Kaggle Dataset, then add that output as Input in the next session. `Save & Run All` starts a fresh notebook execution; it is not a simple snapshot of an already running interactive session.

**Fixed split:** Train Subjects 3–7 (**2,626 samples**); validation Subject 8 (**489 samples**). Test Subjects 1, 2, 9 (**1,695 samples**) are not used here. These counts come from the validated cleaned split, which is rechecked below. This version uses **one GPU** to simplify stable checkpointing and exact resume; distributed training may be added after an end-to-end verification.

## 1 — Experiment configuration (edit here only)

Keep `TOTAL_EPOCHS`, preprocessing, architecture, seed, optimizer, and batch configuration **unchanged across resumed sessions**. Modify only `EPOCHS_THIS_SESSION` or `RESUME_CHECKPOINT_HINT` when necessary.

In [ ]:
from pathlib import Path
import os, sys, re, json, math, time, copy, random, hashlib, shutil, subprocess, tempfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch

# =============== Experiment ===============
SEED = 42
NUM_CLASSES = 38
NUM_FRAMES = 16
INPUT_SIZE = 224
MAX_EVENTS_PER_SAMPLE = 200_000  # Uniformly sample event indices to approximately preserve temporal coverage
RAW_WIDTH, RAW_HEIGHT = 1280, 720

TOTAL_EPOCHS = 60
EPOCHS_THIS_SESSION = 8          # Maximum epochs IN THIS SESSION (does not reset the total)
MAX_SESSION_MINUTES = 450       # Check AFTER each epoch; a process killed mid-epoch cannot be recovered
BATCH_SIZE = 1                  # One GPU; T4 GPU has approximately 16 GB VRAM
ACCUM_STEPS = 8                # Effective batch size = 8; keep unchanged across sessions
NUM_WORKERS = 2
BASE_LR = 1e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 0.05
GRAD_CLIP = 1.0
DROP_PATH = 0.1
FC_DROP_RATE = 0.3
USE_AMP = False                 # FP32 as in the CTP baseline; enable AMP only after validating wheels and kernels
USE_GRAD_CHECKPOINT = False    # May enable before the first session if CUDA OOM occurs; do not change on resume
CHECKPOINT_LAYERS = 12

RUN_NAME = 'egoev_videomamba_tiny_hetero_v1'
OUT_DIR = Path('/kaggle/working') / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_ROOT_HINT = '/kaggle/input/datasets/hoanganh94/egoevgesture-raw'
SPLIT_ROOT_HINT = ''            # Leave blank to discover split_manifest.json automatically
RESUME_CHECKPOINT_HINT = ''    # Leave blank to discover checkpoint-latest.pth in Kaggle Input
PRETRAIN_CHECKPOINT_HINT = ''  # Leave blank to discover or download from Hugging Face if needed
VIDEOMAMBA_SOURCE_HINT = ''    # Leave blank to clone the pinned source repository
REPO_COMMIT = '37355c26d0ae99ca2459f6d4044a5f509031a79f'
REPO_DIR = Path('/kaggle/working/VideoMamba')

if TOTAL_EPOCHS < 1 or EPOCHS_THIS_SESSION < 1: raise ValueError('Epoch count must be > 0')
if INPUT_SIZE != 224 or NUM_FRAMES != 16:
    raise ValueError('The pretrained baseline uses 224x224 and 16 frames; resizing requires positional-embedding validation.')
if WARMUP_EPOCHS >= TOTAL_EPOCHS: raise ValueError('WARMUP_EPOCHS must be smaller than TOTAL_EPOCHS')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

print('Python:',sys.version.split()[0], '| Torch:',torch.__version__, '| CUDA:',torch.version.cuda)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'UNAVAILABLE')
print('Output:',OUT_DIR)

## 2 — Discover Kaggle datasets and verify the split

Fail immediately on missing files, CSV checksum mismatches, or subjects assigned to incorrect splits. **Never use `test.csv` for model training or selection.** The published CSVs store dataset-relative paths.

In [ ]:
INPUT_BASE = Path('/kaggle/input')
if not INPUT_BASE.is_dir(): raise RuntimeError('Run on Kaggle and add the raw dataset and split CSV dataset as Input.')

def find_split_root():
    if SPLIT_ROOT_HINT:
        p=Path(SPLIT_ROOT_HINT)
        if not (p/'train.csv').is_file(): raise FileNotFoundError(p/'train.csv')
        return p
    roots=[]
    for d,dirs,files in os.walk(INPUT_BASE):
        p=Path(d)
        depth=len(p.relative_to(INPUT_BASE).parts)
        if {'train.csv','val.csv','split_manifest.json'} <= set(files):
            roots.append(p)
            dirs[:]=[]
        elif depth>=6 or re.fullmatch(r'subj\d+_(left|right)',p.name):
            dirs[:]=[]
    if len(roots)!=1: raise RuntimeError('Exactly one split dataset with train.csv/val.csv/manifest is required. Found: '+str(roots))
    return roots[0]

def find_raw_root():
    p=Path(RAW_ROOT_HINT)
    if (p/'subj3_left').is_dir() and (p/'subj8_right').is_dir():return p
    roots=[]
    for d,dirs,files in os.walk(INPUT_BASE):
        p=Path(d)
        if (p/'subj3_left').is_dir() and (p/'subj8_right').is_dir():
            roots.append(p);dirs[:]=[]
        elif len(p.relative_to(INPUT_BASE).parts)>=5:dirs[:]=[]
    if len(roots)!=1: raise RuntimeError('Could not uniquely identify the raw dataset. Roots: '+str(roots))
    return roots[0]

SPLIT_ROOT=find_split_root()
RAW_ROOT=find_raw_root()
MANIFEST_PATH=SPLIT_ROOT/'split_manifest.json'
manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))

def sha256_path(path,block=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(block),b''):h.update(b)
    return h.hexdigest()

assert manifest.get('schema_version')=='egoev-clean-split/1.0', 'Sai schema manifest'
assert manifest.get('protocol')=='heterogeneous_subject_holdout_cleaned', 'Sai protocol'
EXPECTED={'train':({3,4,5,6,7},2626),'val':({8},489)}
FRAMES={}
CSV_SHA={}
for name,(subjects,count) in EXPECTED.items():
    p=SPLIT_ROOT/(name+'.csv')
    if not p.is_file():raise FileNotFoundError(p)
    actual=sha256_path(p)
    expected_hash=manifest.get('csv_sha256',{}).get(p.name)
    if actual!=expected_hash: raise RuntimeError('CSV was modified or does not match the manifest: '+p.name)
    df=pd.read_csv(p)
    need={'relative_path','gesture_id','subject','size_bytes'}
    if not need.issubset(df.columns):raise RuntimeError(f'{name}: missing columns {need-set(df.columns)}')
    if len(df)!=count or set(df.subject.astype(int))!=subjects:raise RuntimeError(f'{name}: sai count/subject')
    if set(df.gesture_id.astype(int)) != set(range(38)):raise RuntimeError(f'{name}: missing classes')
    if not df.relative_path.is_unique:raise RuntimeError(f'{name}: duplicate paths')
    if df.gesture_id.isna().any():raise RuntimeError(f'{name}: null label')
    for row in df.itertuples(index=False):
        rel=Path(row.relative_path)
        if rel.is_absolute() or '..' in rel.parts:raise RuntimeError(f'Unsafe relative path: {rel}')
        f=RAW_ROOT/rel
        if not f.is_file() or f.stat().st_size!=int(row.size_bytes):
            raise RuntimeError(f'Sample missing or file size differs from CSV: {f}')
        m=re.fullmatch(r'act(\d+)_(\d+)\.npy',f.name)
        if not m or int(m.group(1))%38!=int(row.gesture_id):raise RuntimeError(f'Sai label: {rel}')
    FRAMES[name]=df
    CSV_SHA[name]=actual
    print(f'{name}: {len(df)} samples, {df.gesture_id.nunique()} classes, subjects={sorted(subjects)} | SHA256={actual[:16]}...')

if set(FRAMES['train'].relative_path)&set(FRAMES['val'].relative_path):
    raise RuntimeError('Train/validation path leakage')
print('RAW_ROOT =',RAW_ROOT)
print('SPLIT_ROOT =',SPLIT_ROOT)
print('SPLIT PREFLIGHT: PASS (Test has not been read or used)')

## 3 — Automatically discover the previous session's checkpoint

Resume only from a `checkpoint-latest.pth` whose configuration and CSV SHA-256 fingerprints match the current experiment. If more than one checkpoint is attached, set `RESUME_CHECKPOINT_HINT` explicitly. Always carry forward the previous session's **best** checkpoint as well.

In [ ]:
IMMUTABLE_CONFIG = {
    'model':'videomamba_tiny', 'num_classes':NUM_CLASSES,
    'num_frames':NUM_FRAMES, 'input_size':INPUT_SIZE, 'max_events':MAX_EVENTS_PER_SAMPLE,
    'sensor_width':RAW_WIDTH,'sensor_height':RAW_HEIGHT,
    'train_csv_sha256':CSV_SHA['train'],'val_csv_sha256':CSV_SHA['val'],
    'seed':SEED,'batch_size':BATCH_SIZE,'accum_steps':ACCUM_STEPS,
    'total_epochs':TOTAL_EPOCHS,'base_lr':BASE_LR,'min_lr':MIN_LR,
    'warmup_epochs':WARMUP_EPOCHS,'weight_decay':WEIGHT_DECAY,
    'grad_clip':GRAD_CLIP,'drop_path':DROP_PATH,'fc_drop_rate':FC_DROP_RATE,
    'use_amp':USE_AMP,'grad_checkpoint':USE_GRAD_CHECKPOINT,
    'checkpoint_layers':CHECKPOINT_LAYERS if USE_GRAD_CHECKPOINT else 0,
    'repo_commit':REPO_COMMIT,
    'event_channels':'positive_count,negative_count,total_count',
    'normalization':'log1p + clip_to_99pct_nonzero + imagenet_rgb_norm',
    'sampling':'linspace_event_index_deterministic',
}
CONFIG_FILE=OUT_DIR/'experiment_config.json'
CONFIG_FILE.write_text(json.dumps(IMMUTABLE_CONFIG,indent=2,ensure_ascii=False),encoding='utf-8')

def locate_resume():
    if RESUME_CHECKPOINT_HINT:
        p=Path(RESUME_CHECKPOINT_HINT)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    candidates=[]
    for root,dirs,files in os.walk(INPUT_BASE):
        p=Path(root)
        if 'checkpoint-latest.pth' in files:
            candidates.append(p/'checkpoint-latest.pth')
            dirs[:]=[]
        elif len(p.relative_to(INPUT_BASE).parts)>=8 or re.fullmatch(r'subj\d+_(left|right)',p.name):
            dirs[:]=[]
    if len(candidates)>1:
        raise RuntimeError('Multiple checkpoint-latest.pth files found; set RESUME_CHECKPOINT_HINT:\n'+'\n'.join(map(str,candidates)))
    return candidates[0] if candidates else None

RESUME_PATH=locate_resume()
if RESUME_PATH:
    print('RESUME checkpoint found:',RESUME_PATH)
    if not (RESUME_PATH.parent/'checkpoint-best.pth').is_file():
        raise FileNotFoundError('Previous session is missing checkpoint-best.pth: '+str(RESUME_PATH.parent))
else:
    print('No resume checkpoint found: initialize from pretrained VideoMamba-Tiny.')

## 4 — Official VideoMamba source and CUDA extension wheels

Use the **OpenGVLab/VideoMamba repository pinned to the same commit as the earlier CTPGesture notebook**. Wheels for `mamba_ssm` and `causal_conv1d` must match the exact **Python ABI, PyTorch, CUDA, and GPU architecture**. This cell does not attempt to build CUDA extensions. If the Kaggle runtime uses Python 3.13 but only `cp312` wheels are available, supply compatible wheels or use a matching runtime. **Do not silently substitute a different model architecture.**

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('GPU not enabled: Kaggle Notebook Settings → Accelerator → GPU T4.')

import importlib, importlib.util, importlib.metadata, sysconfig
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

# If the modules are already installed and importable, wheels are not required.
def cuda_import_ready():
    try:
        from causal_conv1d import causal_conv1d_fn
        from mamba_ssm.modules.mamba_simple import Mamba
        import causal_conv1d_cuda, selective_scan_cuda
        import inspect
        if 'bimamba' not in inspect.signature(Mamba.__init__).parameters:
            raise RuntimeError('mamba_ssm lacks bimamba=True support and is incompatible with this VideoMamba commit.')
        return True
    except Exception as e:
        print('CUDA extension unavailable:',type(e).__name__,str(e)[:400])
        return False

def run_cmd(args, cwd=None):
    print('$',' '.join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),cwd=str(cwd) if cwd else None,check=True)

# Install PyPI dependencies during setup; add prebuilt CUDA wheels as Input.
def installed_version(name):
    try:return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:return None
if installed_version('timm')!='0.4.12' or installed_version('transformers')!='4.36.1' or not importlib.util.find_spec('einops'):
    run_cmd([sys.executable,'-m','pip','install','-q','--upgrade','timm==0.4.12','transformers==4.36.1','einops','packaging'])

if not cuda_import_ready():
    compatible_tags=set(sys_tags())
    wheels=[]
    for root,dirs,files in os.walk(INPUT_BASE):
        p=Path(root)
        for filename in files:
            if filename.endswith('.whl') and (filename.lower().startswith('mamba_ssm-') or filename.lower().startswith('causal_conv1d-')):
                f=p/filename
                try:
                    dist,vers,build,tags=parse_wheel_filename(filename)
                    if tags & compatible_tags:
                        wheels.append((str(dist).replace('-','_').lower(),f))
                except Exception as exc:print('Invalid wheel filename:',filename,exc)
        if len(p.relative_to(INPUT_BASE).parts)>=6 or re.fullmatch(r'subj\d+_(left|right)',p.name):
            dirs[:]=[]
    selection={}
    for package in ('causal_conv1d','mamba_ssm'):
        matches=[f for kind,f in wheels if kind==package]
        if len(matches)!=1:
            raise RuntimeError(f'{package}: found {len(matches)} wheels matching the Python ABI in Kaggle Input. '
                               f'Add a wheel dataset compatible with this runtime ({sys.version.split()[0]}, torch {torch.__version__}, CUDA {torch.version.cuda}); '
                               f'available wheels: {[str(f) for f in matches]}')
        selection[package]=matches[0]
    run_cmd([sys.executable,'-m','pip','install','-q','--no-deps',str(selection['causal_conv1d'])])
    run_cmd([sys.executable,'-m','pip','install','-q','--no-deps',str(selection['mamba_ssm'])])
    importlib.invalidate_caches()
    if not cuda_import_ready():
        raise RuntimeError('Wheels installed but CUDA import failed. Check PyTorch/CUDA/ABI compatibility; training will not continue.')

if VIDEOMAMBA_SOURCE_HINT:
    REPO_DIR=Path(VIDEOMAMBA_SOURCE_HINT)
    if not (REPO_DIR/'videomamba/video_sm/models/videomamba.py').is_file():
        raise FileNotFoundError('VIDEOMAMBA_SOURCE_HINT does not point to the required VideoMamba repository')
else:
    if not (REPO_DIR/'.git').is_dir():
        run_cmd(['git','clone','https://github.com/OpenGVLab/VideoMamba.git',str(REPO_DIR)])
    run_cmd(['git','-C',str(REPO_DIR),'checkout','--detach',REPO_COMMIT])
    revision=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
    if revision!=REPO_COMMIT:raise RuntimeError('Repository revision sai')

VIDEO_SM_DIR=REPO_DIR/'videomamba/video_sm'
if str(VIDEO_SM_DIR) not in sys.path:sys.path.insert(0,str(VIDEO_SM_DIR))
from models.videomamba import videomamba_tiny
print('VIDEO MAMBA IMPORT: PASS',VIDEO_SM_DIR)

## 5 — Convert event streams into 16 three-channel frames

- Read `(N, 4)` events ordered `(x, y, p, t)` from a **1280 × 720** sensor.
- Uniformly subsample up to **200,000 events** across the sequence; split timestamps into **16 temporal bins**.
- Resize to **224 × 224** while preserving aspect ratio with black padding. Channels 0/1/2 contain positive, negative, and total event counts; apply log compression and fixed normalization identically for train, validation, and later evaluation.
- Do not apply horizontal flips: some gesture classes are left/right directional.

This is a **proposed input-representation baseline**, **not** the official event representation from the EgoEvGesture paper. Document it when comparing architectures.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IMAGENET_MEAN=np.array([0.485,0.456,0.406],dtype=np.float32)[:,None,None,None]
IMAGENET_STD=np.array([0.229,0.224,0.225],dtype=np.float32)[:,None,None,None]

def events_to_clip(events, *, num_frames=NUM_FRAMES, out_size=INPUT_SIZE, max_events=MAX_EVENTS_PER_SAMPLE):
    if events.ndim != 2 or events.shape[1]!=4 or len(events)<2:
        raise ValueError(f'Expected event Nx4, >=2 events, got {events.shape}')
    n=len(events)
    idx=np.linspace(0,n-1,min(n,max_events),dtype=np.int64)
    subset=np.asarray(events[idx],dtype=np.float64)
    x,y,p,t=subset.T
    if not (np.isfinite(subset).all()):raise ValueError('NaN/Inf event')
    if not (np.all((0<=x)&(x<RAW_WIDTH)) and np.all((0<=y)&(y<RAW_HEIGHT))):
        raise ValueError('Coordinates outside 1280×720. Check Nx4 column order/sensor geometry.')
    if not (t[-1]>t[0]):raise ValueError('Event timestamps do not span positive time')
    pol=set(np.unique(p).tolist())
    if not pol.issubset({-1.0,0.0,1.0}):raise ValueError(f'Polarity unexpected: {pol}')

    # uniform TIME bins, not equal-event bins
    k=np.floor((t-t[0]) / (t[-1]-t[0]) * num_frames).astype(np.int32)
    k=np.clip(k,0,num_frames-1)
    scale=out_size/max(RAW_WIDTH,RAW_HEIGHT)
    scaled_w=int(round(RAW_WIDTH*scale));scaled_h=int(round(RAW_HEIGHT*scale))
    ox=(out_size-scaled_w)//2;oy=(out_size-scaled_h)//2
    xx=np.clip(np.floor(x*scale).astype(np.int32)+ox,0,out_size-1)
    yy=np.clip(np.floor(y*scale).astype(np.int32)+oy,0,out_size-1)
    pos=(p>0)
    linear=k*(out_size*out_size)+yy*out_size+xx
    bins=num_frames*out_size*out_size
    pos_count=np.bincount(linear[pos],minlength=bins).reshape(num_frames,out_size,out_size)
    neg_count=np.bincount(linear[~pos],minlength=bins).reshape(num_frames,out_size,out_size)
    total=pos_count+neg_count
    clip=np.stack([pos_count,neg_count,total],axis=0).astype(np.float32)
    np.log1p(clip,out=clip)
    nonzero=clip[clip>0]
    denom=float(np.percentile(nonzero,99.0)) if len(nonzero) else 1.0
    clip=np.clip(clip/max(denom,1e-6),0,1)
    clip=(clip-IMAGENET_MEAN)/IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(clip))  # [C,T,H,W]

class EgoEvDataset(Dataset):
    def __init__(self, df,root):
        self.rows=df[['relative_path','gesture_id']].to_records(index=False)
        self.root=Path(root)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        rel,label=self.rows[i]
        p=self.root/str(rel)
        events=np.load(p,mmap_mode='r',allow_pickle=False)
        frames=events_to_clip(events)
        if not torch.isfinite(frames).all():raise ValueError(f'NaN frame: {p}')
        return frames,int(label)

train_dataset=EgoEvDataset(FRAMES['train'],RAW_ROOT)
val_dataset=EgoEvDataset(FRAMES['val'],RAW_ROOT)
for label,ds in [('TRAIN',train_dataset),('VAL',val_dataset)]:
    x,y=ds[0]
    assert x.shape==(3,NUM_FRAMES,INPUT_SIZE,INPUT_SIZE)
    assert 0<=y<NUM_CLASSES
    print(label, '| samples',len(ds),'| example',tuple(x.shape),'| label',y,
          '| tensor min/max',round(float(x.min()),3),round(float(x.max()),3))

def seed_worker(worker_id):
    seed=torch.initial_seed()%2**32
    np.random.seed(seed);random.seed(seed)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,
                        num_workers=NUM_WORKERS,pin_memory=True,
                        persistent_workers=NUM_WORKERS>0,worker_init_fn=seed_worker)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,
                      num_workers=NUM_WORKERS,pin_memory=True,
                      persistent_workers=NUM_WORKERS>0,worker_init_fn=seed_worker)
print('EVENT DATALOADER: PASS')

## 6 — Load pretrained weights or restore a training checkpoint

- **First session:** Load ImageNet-pretrained VideoMamba-Tiny weights (replace the 1,000-class head with a 38-class head). Inflate 2D patch-convolution weights to 3D if needed.
- **Subsequent sessions:** Restore the complete `latest` checkpoint: model, optimizer, learning-rate scheduler, AMP scaler, random-number generator states, and best validation metric. Do **not** initialize from pretrained weights again.
- When resuming, copy the previous `best` checkpoint into the current session's output directory and preserve the best metric.

In [ ]:
from torch import nn
from urllib.request import urlretrieve

PRETRAIN_NAME='videomamba_t16_in1k_res224.pth'
PRETRAIN_URL='https://huggingface.co/OpenGVLab/VideoMamba/resolve/main/'+PRETRAIN_NAME

def locate_pretrained():
    if PRETRAIN_CHECKPOINT_HINT:
        p=Path(PRETRAIN_CHECKPOINT_HINT)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    candidates=[]
    for root,dirs,files in os.walk(INPUT_BASE):
        if PRETRAIN_NAME in files: candidates.append(Path(root)/PRETRAIN_NAME)
        p=Path(root)
        if len(p.relative_to(INPUT_BASE).parts)>=7 or re.fullmatch(r'subj\d+_(left|right)',p.name):dirs[:]=[]
    if len(candidates)>1:raise RuntimeError('Multiple pretrained weight files found; set PRETRAIN_CHECKPOINT_HINT: '+str(candidates))
    if candidates:return candidates[0]
    path=OUT_DIR/PRETRAIN_NAME
    if not path.is_file() or path.stat().st_size<10_000_000:
        print('Downloading pretrained model ...')
        try:urlretrieve(PRETRAIN_URL,path)
        except Exception as exc:raise RuntimeError('Could not download pretrained weights. Enable Internet or add pretrained weights as Input.') from exc
    return path

def safe_load(path):
    try:return torch.load(path,map_location='cpu',weights_only=False)
    except TypeError:return torch.load(path,map_location='cpu')

def make_model():
    return videomamba_tiny(pretrained=False,num_classes=NUM_CLASSES,num_frames=NUM_FRAMES,
                           img_size=INPUT_SIZE,drop_path_rate=DROP_PATH,fc_drop_rate=FC_DROP_RATE,
                           use_checkpoint=USE_GRAD_CHECKPOINT,
                           checkpoint_num=CHECKPOINT_LAYERS if USE_GRAD_CHECKPOINT else 0)

def load_pretrained(model,path):
    ckpt=safe_load(path)
    weights=ckpt.get('model',ckpt.get('state_dict',ckpt))
    if not isinstance(weights,dict):raise RuntimeError('Pretrained checkpoint missing model weights')
    weights={k.removeprefix('module.').removeprefix('backbone.'):v for k,v in weights.items()}
    weights={k:v for k,v in weights.items() if k not in ('head.weight','head.bias')}
    if weights.get('patch_embed.proj.weight') is not None and weights['patch_embed.proj.weight'].ndim==4:
        weights['patch_embed.proj.weight']=weights['patch_embed.proj.weight'].unsqueeze(2)
    target=model.state_dict()
    if 'temporal_pos_embedding' in weights:
        t=weights['temporal_pos_embedding']
        if t.shape!=target['temporal_pos_embedding'].shape:
            weights['temporal_pos_embedding']=F.interpolate(
                t.permute(0,2,1).float(),size=NUM_FRAMES,mode='linear',align_corners=False).permute(0,2,1)
    valid={k:v for k,v in weights.items() if k in target and v.shape==target[k].shape}
    loaded_parameters=sum(v.numel() for v in valid.values())
    all_backbone_parameters=sum(v.numel() for k,v in target.items() if not k.startswith('head.'))
    ratio=loaded_parameters/all_backbone_parameters
    if ratio<0.90:raise RuntimeError(f'Too few compatible pretrained parameters ({ratio:.1%}), aborting')
    ret=model.load_state_dict(valid,strict=False)
    print(f'Loaded pretrained backbone {ratio:.1%}, params={loaded_parameters:,}, '
          f'missing keys={ret.missing_keys[:12]}',flush=True)

model=make_model()
if RESUME_PATH is None:
    pretrained_path=locate_pretrained()
    if pretrained_path.stat().st_size<10_000_000:raise RuntimeError('Pretrained weights too small/invalid')
    load_pretrained(model,pretrained_path)
else:
    pretrained_path=None

DEVICE=torch.device('cuda:0')
model=model.to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=BASE_LR,weight_decay=WEIGHT_DECAY,
                            betas=(0.9,0.999))

def lr_factor(epoch):
    if epoch < WARMUP_EPOCHS:return float(epoch+1)/max(1,WARMUP_EPOCHS)
    progress=(epoch-WARMUP_EPOCHS)/max(1,TOTAL_EPOCHS-WARMUP_EPOCHS-1)
    return MIN_LR/BASE_LR+(1-MIN_LR/BASE_LR)*0.5*(1+math.cos(math.pi*min(progress,1.0)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_factor)
try:
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
except Exception:
    scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)

start_epoch=0
best_acc=-1.0
best_val_loss=float('inf')
history=[]

def check_resume_compat(ckpt):
    expected=ckpt.get('immutable_config')
    if expected!=IMMUTABLE_CONFIG:
        mismatches={k:(expected.get(k) if isinstance(expected,dict) else None, v)
                    for k,v in IMMUTABLE_CONFIG.items() if not isinstance(expected,dict) or expected.get(k)!=v}
        raise RuntimeError('Checkpoint configuration or split CSV does not match: '+str(mismatches))

if RESUME_PATH is not None:
    ckpt=safe_load(RESUME_PATH)
    check_resume_compat(ckpt)
    model.load_state_dict(ckpt['model'],strict=True)
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if USE_AMP:scaler.load_state_dict(ckpt['scaler'])
    best_acc=float(ckpt['best_val_acc'])
    best_val_loss=float(ckpt.get('best_val_loss',float('inf')))
    start_epoch=int(ckpt['epoch'])+1
    history=ckpt.get('history',[])
    # Restore best checkpoint so it persists even if the new session never improves.
    best_previous=safe_load(RESUME_PATH.parent/'checkpoint-best.pth')
    check_resume_compat(best_previous)
    if int(best_previous['epoch'])>int(ckpt['epoch']):
        raise RuntimeError('Best checkpoint is newer than Latest; wrong resume folder.')
    shutil.copy2(RESUME_PATH.parent/'checkpoint-best.pth',OUT_DIR/'checkpoint-best.pth')
    # Restore RNG before new data loader iteration, for reproducible epoch-level continuation.
    rng=ckpt.get('rng')
    if rng:
        random.setstate(rng['python'])
        np.random.set_state(rng['numpy'])
        torch.set_rng_state(rng['torch_cpu'])
        if torch.cuda.is_available() and rng.get('torch_cuda') is not None:
            torch.cuda.set_rng_state_all(rng['torch_cuda'])
    print(f'RESUME PASS: completed {start_epoch}/{TOTAL_EPOCHS} epochs; best val_acc={best_acc:.4f}')
else:
    print('FIRST TRAIN SESSION: starting at epoch 1')

if start_epoch>TOTAL_EPOCHS:
    raise RuntimeError('Checkpoint epoch exceeds TOTAL_EPOCHS')
print('Model params:',sum(p.numel() for p in model.parameters()))
print('LR initial:',optimizer.param_groups[0]['lr'])

## 7 — Forward-pass sanity check (one mini-batch)

If a CUDA out-of-memory error occurs here, do **not** change `NUM_FRAMES` or `INPUT_SIZE` within this fixed experiment. Reduce batch memory requirements, enable gradient checkpointing **before the first session**, or use a GPU with more memory. Changing the model's input geometry defines a different experiment and may make checkpoints incompatible.

In [ ]:
model.eval()
_rng_py=random.getstate(); _rng_np=np.random.get_state(); _rng_torch=torch.get_rng_state()
_rng_cuda=torch.cuda.get_rng_state_all()
with torch.inference_mode():
    xb,yb=next(iter(val_loader))
    out=model(xb.to(DEVICE,non_blocking=True))
    assert out.shape==(xb.shape[0],NUM_CLASSES),out.shape
    assert torch.isfinite(out).all()
print('FORWARD PASS: PASS',out.shape)
del xb,yb,out
random.setstate(_rng_py);np.random.set_state(_rng_np);torch.set_rng_state(_rng_torch)
torch.cuda.set_rng_state_all(_rng_cuda)
torch.cuda.empty_cache()

## 8 — Train and validate; save checkpoints after every epoch

`checkpoint-latest.pth` stores the **complete training state** for resumption. `checkpoint-best.pth` contains the model with the highest validation accuracy. The training history and RNG states are also saved. By default, run no more than `EPOCHS_THIS_SESSION` epochs before a controlled stop. **Do not open `test.csv`.**

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score
from contextlib import nullcontext

def autocast_context():
    return torch.autocast(device_type='cuda',dtype=torch.float16) if USE_AMP else nullcontext()

def train_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    n=0;total_loss=0.;correct=0
    steps=len(train_loader)
    for step,(x,y) in enumerate(train_loader):
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with autocast_context():
            logits=model(x)
            loss=F.cross_entropy(logits,y)
        if not torch.isfinite(loss):raise RuntimeError(f'Non-finite loss at epoch={epoch+1}, step={step}')
        group_remaining=steps-(step//ACCUM_STEPS)*ACCUM_STEPS
        group_size=min(ACCUM_STEPS,group_remaining)
        (scaler.scale(loss/group_size) if USE_AMP else loss/group_size).backward()
        should_step=(step+1)%ACCUM_STEPS==0 or (step+1)==steps
        if should_step:
            if USE_AMP:scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            if USE_AMP:scaler.step(optimizer);scaler.update()
            else:optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        bs=y.numel()
        total_loss+=loss.item()*bs;n+=bs
        correct+=(logits.detach().argmax(1)==y).sum().item()
        if (step+1)%200==0:
            print(f' epoch {epoch+1:03d}, step {step+1}/{steps}, '
                  f'loss={total_loss/n:.4f}, accuracy={correct/n:.3%}',flush=True)
    return total_loss/n,correct/n

@torch.no_grad()
def validate():
    model.eval()
    tot_loss=0.;n=0;y_true=[];y_pred=[]
    for x,y in val_loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with autocast_context():logits=model(x)
        loss=F.cross_entropy(logits,y)
        bs=y.numel();tot_loss+=loss.item()*bs;n+=bs
        y_true.extend(y.cpu().tolist());y_pred.extend(logits.argmax(1).cpu().tolist())
    acc=float(np.mean(np.array(y_true)==np.array(y_pred)))
    f1=float(f1_score(y_true,y_pred,average='macro',labels=list(range(NUM_CLASSES)),zero_division=0))
    return tot_loss/n,acc,f1

def save_checkpoint(path, epoch):
    rng={'python':random.getstate(),'numpy':np.random.get_state(),
         'torch_cpu':torch.get_rng_state(),
         'torch_cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    payload={'format_version':'egoev-videomamba-train-v1',
             'epoch':epoch,'model':model.state_dict(),
             'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
             'scaler':scaler.state_dict() if USE_AMP else None,
             'best_val_acc':best_acc,'best_val_loss':best_val_loss,
             'history':history,'immutable_config':IMMUTABLE_CONFIG,
             'rng':rng,'saved_utc':datetime.now(timezone.utc).isoformat()}
    tmp=path.with_suffix('.tmp')
    torch.save(payload,tmp)
    os.replace(tmp,path)
    print(f'[CHECKPOINT] {path.name} | completed epoch={epoch+1} | size={path.stat().st_size/2**20:.1f} MiB',flush=True)

start_clock=time.monotonic()
end_epoch=min(TOTAL_EPOCHS,start_epoch+EPOCHS_THIS_SESSION)
print(f'Run this session: epochs {start_epoch+1}..{end_epoch} / {TOTAL_EPOCHS}')

for epoch in range(start_epoch,end_epoch):
    lr=optimizer.param_groups[0]['lr']
    tic=time.monotonic()
    tr_loss,tr_acc=train_epoch(epoch)
    val_loss,val_acc,val_f1=validate()
    scheduler.step()
    record={'epoch':epoch+1,'lr':lr,'train_loss':tr_loss,'train_acc':tr_acc,
            'val_loss':val_loss,'val_acc':val_acc,'val_macro_f1':val_f1,
            'epoch_seconds':round(time.monotonic()-tic,1)}
    history.append(record)
    improved=(val_acc>best_acc) or (val_acc==best_acc and val_loss<best_val_loss)
    if improved:
        best_acc=val_acc;best_val_loss=val_loss
        save_checkpoint(OUT_DIR/'checkpoint-best.pth',epoch)
    save_checkpoint(OUT_DIR/'checkpoint-latest.pth',epoch)
    pd.DataFrame(history).to_csv(OUT_DIR/'training_history.csv',index=False)
    print(f'EPOCH {epoch+1}/{TOTAL_EPOCHS} | train {tr_acc:.2%} | '
          f'val {val_acc:.2%} | macro-F1 {val_f1:.4f} | best {best_acc:.2%} '
          f'| elapsed={record["epoch_seconds"]:.1f}s',flush=True)
    if time.monotonic()-start_clock>MAX_SESSION_MINUTES*60:
        print('Session wall-time budget reached AFTER saving checkpoints; stopping safely.');break

if not (OUT_DIR/'checkpoint-latest.pth').exists():
    if RESUME_PATH is None: raise RuntimeError('No epoch completed, checkpoint-latest does not exist.')
    print('No new epoch completed; previous checkpoint still available in Input. Do not publish an empty output.')
print('END TRAIN SESSION, saved files:')
for p in sorted(OUT_DIR.glob('*')):
    if p.is_file():print('  ',p.name,round(p.stat().st_size/2**20,2),'MiB')

## 9 — Verify checkpoints and prepare the next session

**At the end of the session:** Confirm that `checkpoint-latest.pth` and `checkpoint-best.pth` appear in Output. Save the current output as a notebook version or a small Kaggle Dataset containing only `*.pth`, `experiment_config.json`, and `training_history.csv`. In the next session, add this checkpoint output as Input; pretrained weights are no longer required, although the source code and CUDA wheels may still be needed in a new runtime. **Keep Test separate from Train.**

If the runtime terminates before the checkpoint/output is durably saved, training can resume only from the most recent successfully saved checkpoint; incomplete epochs cannot be recovered.

In [ ]:
for name in ('checkpoint-latest.pth','checkpoint-best.pth'):
    path=OUT_DIR/name
    if not path.is_file():
        print(name,': NOT FOUND');continue
    info=safe_load(path)
    check_resume_compat(info)
    print(name, '| completed epochs:',int(info['epoch'])+1,
          '| best val accuracy:',round(float(info['best_val_acc']),4),
          '| contains optimizer:',bool(info.get('optimizer')),
          '| size:',round(path.stat().st_size/2**20,1),'MiB')
    del info
print('For the next session: Add Input from the output containing both checkpoints; keep the configuration unchanged.')

## 10 — Review validation history (without using Test)

In [ ]:
history_path=OUT_DIR/'training_history.csv'
if history_path.is_file():
    from IPython.display import display
    hist=pd.read_csv(history_path)
    display(hist.tail(12))
    ax=hist.plot(x='epoch',y=['train_acc','val_acc'],title='Training/Validation accuracy',grid=True)
    ax.set_ylabel('Accuracy')
    import matplotlib.pyplot as plt
    plt.show()
else:
    print('No full epoch has completed in this session.')

---
### Checklist before using the separate Evaluate notebook

- [ ] The `train.csv` and `val.csv` checksums match the manifest, and both include all 38 classes.
- [ ] `checkpoint-latest.pth` contains the optimizer, scheduler, and RNG state needed for resume.
- [ ] `checkpoint-best.pth` was selected **using validation only**.
- [ ] The previous session's checkpoints were **durably saved** (Kaggle Notebook Output or Kaggle Dataset) before terminating that session.
- [ ] All comparison models use the **same `split_manifest.json`**, and the same preprocessing when isolating architectural differences.
- [ ] `test.csv` is opened only in the final evaluation notebook.

**Verification status:** Notebook structure, CSV handling, event preprocessing, and checkpoint logic can be checked statically or with mock data outside Kaggle. Real GPU Mamba CUDA execution and the full 207 GB dataset have not been tested in this authoring environment. Run the GPU forward-pass sanity check before committing extensive GPU time.